# Lab 5.1 · Singular Directions and Low-Rank Approximation

Build a compact SVD, then predict the error caused by dropping a singular direction.

Wooseok Ha, MAS110 Lecture 5, printed slides 3–23, 27–42, and 45–48. Example 5.3 is on slides 35–36, and its rank-two approximation is revisited on slide 54.

Lecture 5 begins with observations $a_i\in\mathbb{R}^d$ stored as rows of $A$. Projecting a row onto a unit direction $v$ gives the coordinate $a_i^{\top}v$. Stacking these coordinates gives $Av$. Pythagoras turns the smallest sum of squared residual lengths into the largest sum of squared projected lengths.

Run the cells in order with NumPy. The code and notation follow the web lab.

## Notation

- $A\in\mathbb{R}^{n\times d}$ · $n$ counts rows and $d$ counts features. Right singular vectors live in feature space and left singular vectors live in the space of row coordinates.

- $(\sigma_i,v_i,u_i)$ · $\sigma_i=\lVert Av_i\rVert_2$ and $Av_i=\sigma_i u_i$. A triplet with $\sigma_i>0$ pairs a unit input direction and its unit output direction.

- $A=UDV^{\top}$ · In the lecture’s compact SVD, $r$ is the exact rank, $U$ has shape $n\times r$, $D$ has shape $r\times r$, and $V$ has shape $d\times r$. NumPy’s reduced SVD returns $p=\min(n,d)$ slots, including any zero or numerically tiny ones. Its `s` stores diagonal entries and `Vt` stores right singular vectors as rows.

- $w_i$ and $v_i$ · In the example, $w_i$ denotes the eigenvector labeled $v_i$ on slide 35. The lab uses $v_i$ for the right singular vectors in decreasing singular-value order.

## 1. Order the scales and pair the vectors

The eigenvalues in singular-value order are $-6$, $4$, and $2$. Enter their eigenvectors from slide 35 in that order.

**Predict first.** Which column of $U$ changes sign relative to $V$?

In [ ]:
import numpy as np
A = np.array([[-1., -5., 0., 0.], [-5., -1., 0., 0.], [0., 0., 2., 0.], [0., 0., 0., 0.]])
h = 1 / np.sqrt(2)
V = np.array([[h, h, 0.], [h, -h, 0.], [0., 0., 1.], [0., 0., 0.]])
eigenvalues = np.array([-6., 4., 2.])
s = np.abs(eigenvalues)
U = V * np.sign(eigenvalues)
print("D diagonal =", s, "\nU =", U, "\nV =", V)

The first left direction is the negative of its paired right direction.

## 2. Rebuild the matrix

Each column of $U$ gets multiplied by its singular value. Multiplying by $V^{\top}$ then adds the three outer products.

**Predict first.** Will the zero fourth row change?

In [ ]:
D = np.diag(s)
rebuilt = U @ D @ V.T
U_lib, s_lib, Vt_lib = np.linalg.svd(A, full_matrices=False)
print("Reconstructed A =", rebuilt, "\nNumPy singular values =", s_lib)

The compact reconstruction equals the original. NumPy also returns a zero singular value for this example.

$$A=UDV^{\top}$$

## 3. Retain the first two triplets

Slice the factors consistently. Both kept directions lie in the first two coordinates, so the third row and column become zero.

**Predict first.** Which singular value survives in the error matrix?

In [ ]:
k = 2
A_k = (U[:, :k] * s[:k]) @ V[:, :k].T
E = A - A_k
errors = [np.linalg.norm(E, 2), np.linalg.norm(E, "fro")]
print("Rank-two A =", A_k, "\nError =", E, "\nSpectral and Frobenius norms =", errors)

The remaining stretch is two in the third direction.

$$\lVert E\rVert_2=\lVert E\rVert_F=2$$

## Why these directions work

### The best line gives the first right singular vector

The lecture first chooses the unit $v_1$ maximizing $\lVert Av\rVert_2$. It then chooses the best unit direction orthogonal to all earlier ones. Theorem 5.1 makes this greedy choice exact for the squared projection objective. For a plane, any competing plane contains a unit direction orthogonal to $v_1$. Complete it to an orthonormal basis and compare each basis direction with the corresponding greedy maximum.

$$W_k=\operatorname{span}\{v_1,\ldots,v_k\}$$

### Move the eigenvalue sign into the left vector

For a real symmetric matrix we can start with orthonormal eigenvectors and order them by decreasing $|\lambda_i|$. Taking $\sigma_i=|\lambda_i|$ and $u_i=\operatorname{sign}(\lambda_i)v_i$ preserves the action on each positive-scale direction. In Example 5.3 the direction with eigenvalue $-6$ comes first and its output points the opposite way.

$$Av_i=\sigma_i u_i,\quad A^{\top}u_i=\sigma_i v_i$$

### Read both norms from the discarded terms

The rank-one terms use orthonormal directions on both sides. The spectral norm is the largest stretch. The squared Frobenius norm is the sum of the squared singular values. Eckart–Young–Mirsky says that keeping the largest $k$ terms minimizes both errors among matrices of rank at most $k$. In slide 54’s middle expression, the second kept term is $4w_3w_3^{\top}$. This agrees with the matrix printed on the slide.

$$\lVert A-A_k\rVert_2=\sigma_{k+1},\quad \lVert A-A_k\rVert_F^2=\sum_{i=k+1}^{r}\sigma_i^2$$

## Verify the example

Run these checks on the example before changing the input.

In [ ]:
assert np.allclose(U.T @ U, np.eye(3))
assert np.allclose(V.T @ V, np.eye(3))
assert np.allclose(A @ V, U * s)
assert np.allclose(A.T @ U, V * s)
assert np.allclose(rebuilt, A)
assert np.allclose((U_lib * s_lib) @ Vt_lib, A)
assert np.allclose(s_lib, [6, 4, 2, 0])
assert np.linalg.matrix_rank(A_k) == 2
assert np.allclose(errors, [2, 2])
E1 = A - s[0] * np.outer(U[:, 0], V[:, 0])
assert np.isclose(np.linalg.norm(E1, 2), 4)
assert np.isclose(np.linalg.norm(E1, "fro"), np.sqrt(20))

## Centered PCA

Reproduce the example on the chapter page. Centering moves the fit through the mean in the original coordinates. The sign of the scores can vary with the sign of the returned direction.

In [ ]:
observations = np.array([[5., 2.], [1., 2.], [3., 3.], [3., 1.]])
mean = observations.mean(axis=0)
centered = observations - mean
U_pca, s_pca, Vt_pca = np.linalg.svd(centered, full_matrices=False)
v1 = Vt_pca[0]
scores = centered @ v1
fit = mean + np.outer(scores, v1)
variance = s_pca**2 / (len(observations) - 1)
assert np.allclose(mean, [3, 2])
assert np.allclose(variance, [8 / 3, 2 / 3])
assert np.isclose(variance[0] / variance.sum(), 0.8)
assert np.allclose(fit, [[5, 2], [1, 2], [3, 2], [3, 2]])
print("Scores =", scores)
print("Fitted points =", fit)
print("Explained fraction =", variance[0] / variance.sum())

## Check your understanding

Why can NumPy return different vector signs?

Flipping both $u_i$ and $v_i$ leaves their outer product unchanged. With tied singular values the basis within the tied subspace can also vary. Compare reconstructed matrices and invariant subspaces.

What are the two errors if you keep only the first term?

The tail contains $4$ and $2$. Its spectral norm is $4$ and its Frobenius norm is $\sqrt{4^2+2^2}=\sqrt{20}$.

## References

- [NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [Original Chapter 5.1 notebook](https://github.com/kyunghyuncho/Foundations_of_LADS/blob/main/Ch5-1%20SVD-Basic%20Properties.ipynb)